<a href="https://colab.research.google.com/github/Lucianadeoliveira/gaTE-lab/blob/main/FRALCAT/ANALYSIS_AND_PROTEIN_INFERENCE/StatisticalAnalysisFRALCAT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


Statistical Analysis FRALCAT
==============================================================

This script performs statistical analyses on protein sequence
datasets, including:
  - Histograms of amino acid counts
  - Histograms of MSA lengths
  - Scatter plots of sequence identity vs. number of amino acids
  - Box plots for grouped comparisons

All plots are saved automatically to the specified output paths.

Author: Luciana Renata de Oliveira
Date: 29/08/2025
==============================================================


In [ ]:
# ==============================
# INITIAL CONFIGURATION (inputs/outputs)
# ==============================

from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Base directories
BASE_DIR = '/content/drive/MyDrive/GateLab-ProjetoTT5- Marie Anne/'
RESULTS_DIR = os.path.join(BASE_DIR, 'resultados/04-11-24-Resultados-Nova-montagem-Estatísticas')

# Input CSV files
SEQIDENTITY_FILE = os.path.join(RESULTS_DIR, 'All-data/seqIdentityNumberAAAll.csv')

# ==============================
# IMPORTS
# ==============================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ==============================
# FUNCTION 1: Amino Acid Histogram
# ==============================
def plot_histogram_aminoacids(file_path, output_path, color='orange'):
    """
    Generate a histogram showing the distribution of amino acid counts per protein sequence.

    Parameters
    ----------
    file_path : str
        Path to the CSV file containing the column 'Number of Amino Acids'.
    output_path : str
        Path where the histogram will be saved as an image.
    color : str, optional
        Color of the histogram bars (default is 'orange').
    """
    data = pd.read_csv(file_path)
    max_value = data['Number of Amino Acids'].max()
    bin_edges = list(range(0, max_value + 101, 100))

    plt.figure(figsize=(10, 6))
    n, bins, _ = plt.hist(data['Number of Amino Acids'], bins=bin_edges,
                          color=color, edgecolor='black', alpha=0.7)

    for count, x in zip(n, (bins[:-1] + bins[1:]) / 2):
        plt.text(x, count, str(int(count)), ha='center', va='bottom', fontsize=10)

    plt.xlabel('Number of Amino Acids')
    plt.ylabel('Frequency')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    plt.savefig(output_path)
    plt.close()

# ==============================
# FUNCTION 2: MSA Length Histogram
# ==============================
def plot_msa_distribution(base_dir, output_path, color='#0cdc73'):
    """
    Generate a histogram with the distribution of MSA lengths
    collected across all subdirectories.

    Parameters
    ----------
    base_dir : str
        Base directory to recursively search for 'lengthMSAresults.csv' files.
    output_path : str
        Path where the histogram will be saved as an image.
    color : str, optional
        Color of the histogram bars (default is '#0cdc73').
    """
    collected_values = []
    for root, dirs, files in os.walk(base_dir):
        for file in files:
            if file == 'lengthMSAresults.csv':
                df = pd.read_csv(os.path.join(root, file))
                collected_values.extend(df.iloc[1:, 1].values)

    collected_values = pd.to_numeric(collected_values, errors='coerce')
    collected_values = collected_values[~np.isnan(collected_values)]

    plt.figure(figsize=(10, 6))
    n, bins, _ = plt.hist(collected_values, bins=20, color=color, edgecolor='black')

    for count, x in zip(n, (bins[:-1] + bins[1:]) / 2):
        plt.text(x, count, str(int(count)), ha='center', va='bottom', fontsize=10)

    plt.xlabel("MSA Length")
    plt.ylabel("Frequency")
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    plt.savefig(output_path)
    plt.close()

# ==============================
# FUNCTION 3: Scatter Plot
# ==============================
def plot_scatter(data, x, y, output_path):
    """
    Generate a scatter plot between two variables.

    Parameters
    ----------
    data : DataFrame
        Data containing the columns of interest.
    x : str
        Column name for x-axis.
    y : str
        Column name for y-axis.
    output_path : str
        Path where the scatterplot will be saved.
    """
    plt.figure(figsize=(8, 6))
    sns.scatterplot(x=x, y=y, data=data, alpha=0.7)
    plt.xlabel(x)
    plt.ylabel(y)
    plt.grid(True)
    plt.tight_layout()
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    plt.savefig(output_path)
    plt.close()

# ==============================
# FUNCTION 4: Box Plot
# ==============================
def plot_boxplot(data, x, y, output_path):
    """
    Generate a boxplot to compare distributions.

    Parameters
    ----------
    data : DataFrame
        Data containing the columns of interest.
    x : str
        Column name for grouping variable (categorical).
    y : str
        Column name for values (numerical).
    output_path : str
        Path where the boxplot will be saved.
    """
    plt.figure(figsize=(10, 6))
    sns.boxplot(x=x, y=y, data=data)
    sns.stripplot(x=x, y=y, data=data, color='black', jitter=0.2, size=5)
    plt.xlabel(x)
    plt.ylabel(y)
    plt.grid(True)
    plt.tight_layout()
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    plt.savefig(output_path)
    plt.close()

# ==============================
# MAIN EXECUTION
# ==============================
if __name__ == "__main__":
    # Histogram of amino acids
    plot_histogram_aminoacids(
        SEQIDENTITY_FILE,
        os.path.join(RESULTS_DIR, "histogram_number_of_amino_acids.png")
    )

    # Histogram of MSA lengths
    plot_msa_distribution(
        BASE_DIR,
        os.path.join(RESULTS_DIR, "histogram_msa_length.png")
    )

    # Scatter plot: Number of Amino Acids vs. Max Sequence Identity
    data = pd.read_csv(SEQIDENTITY_FILE)
    plot_scatter(
        data,
        x="Number of Amino Acids",
        y="Max Value Sequence Identity",
        output_path=os.path.join(RESULTS_DIR, "scatter_seq_identity.png")
    )

    # Box plot: grouped by AA sequence length quantiles
    data['AA Group'] = pd.qcut(data['Number of Amino Acids'], 10, duplicates='drop')
    plot_boxplot(
        data,
        x="AA Group",
        y="Max Value Sequence Identity",
        output_path=os.path.join(RESULTS_DIR, "boxplot_seq_identity.png")
    )

    print("All statistical analyses completed and saved successfully.")


Mounted at /content/drive
All statistical analyses completed and saved successfully.
